# Step 1: Install necesscary packages

In [13]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


# Step 2: Package imports and configuration


In [14]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
#1os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
from model import GPT, GPTConfig

import torch

# Check if GPU is available
print("CUDA available:", torch.cuda.is_available())

# Check which device is being used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Optional: print GPU name
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
Using device: cuda
GPU name: NVIDIA GeForce RTX 3060 Ti


In [15]:
# Configuration

beta = 0.30

base_lr = 9e-5 

weight_decay = 0.01

epochs = 5

batch_size = 64

max_length = 64

num_samples = 1

max_new_tokens = 200

temperature = 1e-8

top_k = 20


In [16]:
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

# Step 3: Define helper functions

In [17]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

# Step 4: Load the pretrained NanoGPT model

In [18]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

# Step 5: Load Data (students are required to complete this part!)

In [19]:
def clean_text(s, allowed_chars):
    return ''.join([c for c in s if c in allowed_chars])

import json
import tiktoken
lines = ""
with open("../pos_neg_pairs1.json", "r", encoding = "utf-8") as f:
    lines = json.load(f)
    print(f"Loaded {len(lines)} pairs.")

# Clean dataset
allowed_chars = set(stoi.keys())
cleaned_lines = []
removed_count = 0

for pair in lines:
    neg = clean_text(pair['negative'], allowed_chars)
    pos = clean_text(pair['positive'], allowed_chars)

    # Skip pairs that become empty or too short
    if len(pos.strip()) < 2 or len(neg.strip()) < 2:
        removed_count += 1
        continue

    cleaned_lines.append({'negative': neg, 'positive': pos})

print(f"Cleaned dataset: {len(cleaned_lines)} valid pairs (removed {removed_count})")

lines = cleaned_lines

from sklearn.model_selection import train_test_split
train_lines, val_lines = train_test_split(lines, test_size=0.1, random_state=42)
lines = train_lines 
print(f"Train: {len(train_lines)} | Validation: {len(val_lines)}")

Loaded 100000 pairs.
Cleaned dataset: 100000 valid pairs (removed 0)
Train: 90000 | Validation: 10000


# Step 6: Build the optimizer and scheduler (students are required to complete this part!)

In [20]:
import math 
optimizer = gpt.configure_optimizers(weight_decay, base_lr, (0.9, 0.95), device)

def scheduler(total_steps, base_lr, current_step):
    max_lr = base_lr
    min_lr = 0.1 * base_lr
    warmup_steps = round(0.03 * total_steps)

    if current_step <= warmup_steps:
        lr = max_lr * (current_step / warmup_steps)

    elif current_step >= total_steps:
        lr = min_lr

    else:
        p = (current_step - warmup_steps) / (total_steps - warmup_steps)
        lr = min_lr + 0.5*(max_lr - min_lr)*(1 + math.cos(math.pi * p))

    return lr 

num decayed parameter tensors: 26, with 8,834,328 parameters
num non-decayed parameter tensors: 13, with 4,524 parameters
using fused AdamW: False


# Step 7: Begin training (students are required to complete this part!)

In [ ]:
# mixed precision training setup

# AMP policy: prefer bf16 if available; else use fp16 with GradScaler
use_cuda = (device == 'cuda')
use_bf16 = False
autocast_dtype = torch.float16

if use_cuda:
    major_cc, minor_cc = torch.cuda.get_device_capability(0)
    # Ampere (SM80) or newer support bf16 well; some Turing also partially supports but Ampere+ is safer
    if major_cc >= 8:
        use_bf16 = True
        autocast_dtype = torch.bfloat16
    else:
        use_bf16 = False
        autocast_dtype = torch.float16

use_amp = use_cuda  # only enable autocast on CUDA
# GradScaler not needed for bf16; needed for fp16
scaler = torch.amp.GradScaler('cuda', enabled=use_amp and not use_bf16)

print(f"AMP enabled: {use_amp}, dtype: {autocast_dtype}, using GradScaler: {scaler.is_enabled() if use_amp else False}")

AMP enabled: False, dtype: torch.float16, using GradScaler: False


In [ ]:
# helper: validation evaluation function

@torch.no_grad()
def evaluate_validation_loss(model, val_lines, batch_size=32):
    model.eval()
    total_loss, count = 0.0, 0
    for neg_tensor, pos_tensor in get_batches(val_lines, batch_size):
        # Full precision eval for stable metrics
        neg_log = compute_logprob(neg_tensor)
        pos_log = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_log - neg_log) / beta).mean() - 0.1 * pos_log.mean()
        total_loss += loss.item()
        count += 1
    model.train()
    return total_loss / max(count, 1)

In [24]:
# begin training
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optimizer.zero_grad(set_to_none=True)

        # Forward + loss under autocast for speed
        with torch.amp.autocast(dtype=autocast_dtype, enabled=use_amp, device_type='cuda'):
            neg_log = compute_logprob(neg_tensor)
            pos_log = compute_logprob(pos_tensor)
            loss = -F.logsigmoid((pos_log - neg_log) / beta).mean() - 0.1 * pos_log.mean()

        global_step = epoch * total_steps + step + 1
        lr = scheduler(total_steps * epochs, base_lr, global_step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        if scaler.is_enabled():
            # fp16 path with scaling
            scaler.scale(loss).backward()
            # unscale before clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            # bf16 or CPU path
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
            optimizer.step()

        pbar.set_description(f"epoch {epoch+1} step {step+1} train={loss.item():.4f} lr={lr:.2e}")

    #val_loss = evaluate_validation_loss(gpt, val_lines, batch_size=batch_size)
    #print(f"Validation loss after epoch {epoch+1}: {val_loss:.4f}")

    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

epoch 1 step 1406 train=0.0286 lr=8.40e-05: : 1406it [02:06, 11.14it/s]


Saved checkpoint to ./dpo.pt


epoch 2 step 1406 train=0.0243 lr=6.42e-05: : 1406it [02:05, 11.21it/s]


Saved checkpoint to ./dpo.pt


epoch 3 step 1406 train=0.0220 lr=3.85e-05: : 1406it [02:05, 11.22it/s]


Saved checkpoint to ./dpo.pt


epoch 4 step 1406 train=0.0212 lr=1.72e-05: : 1406it [02:05, 11.22it/s]


Saved checkpoint to ./dpo.pt


epoch 5 step 1406 train=0.0208 lr=9.00e-06: : 1406it [02:05, 11.22it/s]


Saved checkpoint to ./dpo.pt


# Step 8: Begin testing (students are required to complete this part!)

In [25]:
# testing

# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

test_set = [
    "12+9=?",
    "33-11=?",
    "9*7=?",
    "64/8=?",
    "99-83=?",
    "25+79=?",
    "5*18=?",
    "60/12=?",
    "2*18=?",
    "45+69=?",

    # x equations
    "x+12=28,x=?",
    "x-9=37,x=?",
    "x*8=64,x=?",
    "x/4=11,x=?",
    "x+17=45,x=?",
    "x-11=90,x=?",
    "x*9=81,x=?",
    "x/2=14,x=?",
    "x*7=84,x=?",
    "x+33=99,x=?",

    # reverse-style equations
    "45+x=90,x=?",
    "72-x=63,x=?",
    "18/x=3,x=?",
    "7*x=28,x=?",
    "81/x=9,x=?",
    "6*x=54,x=?",
    "13+x=27,x=?",
    "99-x=1,x=?",
    "48/x=2,x=?",
    "20/x=4,x=?"
]

with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        prompt_ids_formatted = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        result = gpt.generate(prompt_ids_formatted, max_new_tokens, temperature, top_k)
        decoded_result = decode(result[0].detach().cpu().view(-1).tolist())
        print(decoded_result)
        ###########################################################

12+9=? The answer is 21 because 12+9 equals 21.
33-11=? The answer is 22 because 33-11 equals 22.
9*7=? The answer is 63 because 9*7 equals 63.
64/8=? The answer is 8 because 64/8 equals 8.
99-83=? The answer is 16 because 99-83 equals 16.
25+79=? The answer is 104 because 25+79 equals 104.
5*18=? The answer is 90 because 5*18 equals 90.
60/12=? The answer is 5 because 60/12 equals 5.
2*18=? The answer is 36 because 2*18 equals 36.
45+69=? The answer is 114 because 45+69 equals 114.
x+12=28,x=? The answer is 16 because 28-12 equals 16.
x-9=37,x=? The answer is 46 because 37+9 equals 46.
x*8=64,x=? The answer is 8 because 64/8 equals 8.
x/4=11,x=? The answer is 44 because 4*11 equals 44.
x+17=45,x=? The answer is 28 because 45-17 equals 28.
x-11=90,x=? The answer is 101 because 90+11 equals 101.
x*9=81,x=? The answer is 9 because 81/9 equals 9.
x/2=14,x=? The answer is 36 because 2*14 equals 36.
x*7=84,x=? The answer is 12 because 84/7 equals 12.
x+33=99,x=? The answer is 66 because 99-

In [ ]:
# testing

# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

test_set = [
    "88+7=?",
    "x-18=21,x=?",
    "x/10=6,x=?",
    "54/1=?",
    "24+48=?",
    "11+23=?",
    "64+13=?",
]

with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        prompt_ids_formatted = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        result = gpt.generate(prompt_ids_formatted, max_new_tokens, temperature, top_k)
        decoded_result = decode(result[0].detach().cpu().view(-1).tolist())
        print(decoded_result)
        ###########################################################